# Deep learning on images

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [3]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-03 17:12:06.198907: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-03 17:12:06.240796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-03 17:12:07.186294: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1759504327.825494   13171 gpu_device.cc:2020] Created device /job:localhost/rep

In [ ]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

load_model=True
# load_model=False

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if rebalance_with_weights:
    print('using class weights')
else:
    print('not using class weights')

using class weights


In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(6793, 31)


In [17]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [23]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    subversion = last_experiment.get('subversion', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Création d'un nouveau modèle.


In [24]:
if not load_model:
    subversion = int(input(f"subversion (architecture)? (last: {subversion})"))

In [25]:
subversion

4

### Summary

In [26]:
# model.summary()

## Callbacks

### ModelCheckpoint

In [27]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [28]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [29]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [32]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [34]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
tensor_board_folder_timestamp = tensor_board_folder / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder_timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [35]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [36]:
# max_epochs=13

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [42]:
# Pick max_epochs based on your available time
available_minutes=12

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

5

### compilation and callbacks

In [43]:
learning_rate=0.001

In [44]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [45]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [46]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=12, max_epochs=5, champion_path=None ?

In [47]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=max(total_epochs_trained-1,0), callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/5


2025-10-03 17:12:35.304528: I external/local_xla/xla/service/service.cc:163] XLA service 0x76e448002520 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-03 17:12:35.304556: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-03 17:12:35.621908: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-03 17:12:36.929662: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-03 17:12:37.850790: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 17:12:37.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.1301 - loss: 3.4188

2025-10-03 17:13:41.770784: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12664', 12 bytes spill stores, 12 bytes spill loads

2025-10-03 17:13:45.841155: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 17:13:45.935569: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 17:13:46.553156: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 17:13:46.652745: E external/local_xla/xla/s

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.1303 - loss: 3.4176

2025-10-03 17:15:04.587179: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 17:15:12.557409: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 17:15:12.655798: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 17:15:13.604723: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 173s 662ms/step - accuracy: 0.1886 - loss: 3.1630 - val_accuracy: 0.3586 - val_loss: 2.5539 - learning_rate: 0.0010
Epoch 2/5
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 481ms/step - accuracy: 0.3227 - loss: 2.6185 - val_accuracy: 0.4560 - val_loss: 2.2024 - learning_rate: 0.0010
Epoch 3/5
213/213 ━━━━━━━━━━━━━━━━━━━━ 103s 483ms/step - accuracy: 0.3686 - loss: 2.4264 - val_accuracy: 0.4789 - val_loss: 2.0722 - learning_rate: 0.0010
Epoch 4/5
213/213 ━━━━━━━━━━━━━━━━━━━━ 104s 489ms/step - accuracy: 0.4028 - loss: 2.2916 - val_accuracy: 0.4892 - val_loss: 2.0068 - learning_rate: 0.0010
Epoch 5/5
213/213 ━━━━━━━━━━━━━━━━━━━━ 104s 489ms/step - accuracy: 0.4381 - loss: 2.1867 - val_accuracy: 0.5139 - val_loss: 1.9397 - learning_rate: 0.0010


'total_minutes=9.774985619386037'

## Evaluation

In [48]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [49]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5138954520225525,
 'actual_epochs': 5,
 'minutes_per_epoch': 1.9549971238772073}

In [50]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [51]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 72s 126ms/step


array([[1.9960566e-03, 4.4445633e-03, 1.2026837e-02, ..., 3.5898875e-02,
        2.0706793e-03, 2.8457751e-03],
       [8.1178201e-03, 7.1822535e-03, 1.8154031e-02, ..., 1.5826678e-02,
        2.0725348e-03, 4.2587100e-03],
       [5.1489525e-04, 5.5288309e-03, 5.5194829e-02, ..., 1.2813093e-01,
        3.3875459e-04, 7.2300568e-04],
       ...,
       [2.0454872e-01, 1.8828744e-02, 1.5813220e-04, ..., 9.0409587e-05,
        4.7481349e-01, 1.6833935e-02],
       [5.6012454e-03, 4.8538395e-03, 7.2554420e-03, ..., 1.7620308e-02,
        1.7107893e-03, 1.3154796e-03],
       [1.9676235e-02, 3.3458151e-02, 2.2943363e-02, ..., 4.9388461e-02,
        7.8612128e-03, 3.0120728e-03]], shape=(16984, 27), dtype=float32)

In [52]:
from sklearn import metrics

In [53]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [54]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1280,1281,1300,1302,1320,1560,1920,2060,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,
10,208,8,0,0,4,29,3,0,1,0,1,1,3,8,142,108,0,9,0,1,0,97,0
40,19,91,0,2,17,99,3,0,50,0,2,7,2,8,97,19,8,8,0,16,0,54,0
50,1,11,9,25,11,22,14,0,129,0,0,9,1,12,5,9,10,12,0,55,0,1,0
60,0,4,0,76,0,14,1,0,44,0,0,0,0,5,5,2,6,3,0,4,0,0,2
1140,2,14,0,0,326,42,45,1,11,1,6,2,3,12,19,9,4,3,0,27,0,7,0
1160,1,34,0,0,8,689,1,1,0,0,0,0,0,2,40,12,1,0,0,1,0,1,0
1180,0,1,0,0,49,27,6,2,3,0,4,3,1,10,16,13,1,3,0,12,0,2,0
1280,4,11,1,5,111,25,331,5,210,5,16,24,16,88,18,12,3,12,0,70,1,5,1
1281,12,15,0,1,12,89,61,8,20,6,3,5,1,39,28,42,9,18,0,25,1,15,4


In [55]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.4594376012796143, 0.0)

In [56]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [57]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

,precision,recall,f1-score,support
10,0.452174,0.333868,0.384118,623.000000
40,0.350000,0.181275,0.238845,502.000000
50,0.500000,0.026786,0.050847,336.000000
60,0.633333,0.457831,0.531469,166.000000
1140,0.490964,0.610487,0.544240,534.000000
1160,0.547260,0.871049,0.672195,791.000000
1180,0.000000,0.000000,0.000000,153.000000
1280,0.380023,0.339836,0.358808,974.000000
1281,0.363636,0.019324,0.036697,414.000000
1300,0.470381,0.794846,0.591010,1009.000000


In [58]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.422093,0.384723,0.362201,629.037037
std,0.226123,0.323202,0.267750,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.356818,0.029457,0.054515,310.000000
50%,0.484507,0.339836,0.384118,534.000000
75%,0.553312,0.667312,0.584615,953.500000
max,0.812903,0.871049,0.765957,2042.000000


In [59]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.665829,0.775085,0.061439
recall,0.665829,1.000000,0.969505,0.105956
f1-score,0.775085,0.969505,1.000000,0.093061
support,0.061439,0.105956,0.093061,1.000000


In [60]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.4594376012796143

In [61]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5138954520225525,
 'actual_epochs': 5,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.4594376012796143,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.26775042205936267)}

## Update tracker

In [62]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmin(model_history.history['val_loss'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_loss = model_history.history['val_loss'][best_epoch_in_session_idx]


    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epoch_index-{best_epoch_global:02d}_val_loss-{best_val_loss:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même subversion {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras


In [63]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

In [64]:
to_track=['subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5138954520225525,
 'actual_epochs': 5,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.4594376012796143,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.26775042205936267),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras',
 'epoch_index': np.int64(4),
 'total_epochs': np.int64(5),
 'subversion': 4,
 'max_epochs': 5,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 32,
  'image_dense': 64,
  'tabular_dense_2': 16,
  'final_dense_1': 128},
 'embedding_dims': {'pHash_embedding': 8, 'md5_embedding': 8}}

In [71]:
tracker['comment']="Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. No overfitting: performance is better on validation than on train! And learning rate has not been reduced by ReduceLROnPlateau! Encouraging. Must resume training of this model later."
tracker['comment']

'Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. No overfitting: performance is better on validation than on train! And learning rate has not been reduced by ReduceLROnPlateau! Encouraging. Must resume training of this model later.'

In [72]:
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5138954520225525,
 'actual_epochs': 5,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.4594376012796143,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.26775042205936267),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras',
 'epoch_index': np.int64(4),
 'total_epochs': np.int64(5),
 'subversion': 4,
 'max_epochs': 5,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 32,
  'image_dense': 64,
  'tabular_dense_2': 16,
  'final_dense_1': 128},
 'embedding_dims': {'pHash_embedding': 8, 'md5_embedding': 8},
 'comment': 'Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. No overfitting: performance is better on validation than on train! And learning rate has not been reduced by ReduceLROnPlateau! Encouraging. Must resume training of this model

In [73]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [74]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [75]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [76]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, log_file_path=log_file_path)

Log pour l'expérience subversion 4 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [77]:
pd.set_option('max_colwidth', None)

In [78]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment,best_model_path
0,1,False,6793,32,2.024143,9,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which indicates overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
1,2,False,6793,32,1.998601,19,10,0.001,0.569713,0.528593,0.0,0.247884,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
2,2,False,20380,32,3.867331,9,8,0.001,0.584609,0.563841,0.0,0.239801,256_128_64_32,16_16,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras
3,3,False,6793,32,2.024143,9,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
4,4,True,6793,32,1.954997,5,5,0.001,0.513895,0.459438,0.0,0.267750,128_64_32_16,8_8,Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. No overfitting: performance is better on validation than on train! And learning rate has not been reduced by ReduceLROnPlateau! Encouraging. Must resume training of this model later.,artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.


In [ ]:
print(tracking_df)

   subversion  rebalance_with_weights  X_train.shape[0]  BATCH_SIZE  \
0           1                   False             20380          32   
1           2                   False              6793          32   
2           2                   False             20380          32   

   minutes_per_epoch  max_epochs  total_epochs  learning_rate  val_accuracy  \
0           2.657786           8            13          0.001      0.608220   
1           1.998601          19            10          0.001      0.569713   
2           3.867331           9             8          0.001      0.584609   

   weighted_avg_f1_score  min_f1_score  std_f1_score dense_layers_sizes  \
0               0.598100      0.254545      0.180580      256_128_64_32   
1               0.528593      0.000000      0.247884      256_128_64_32   
2               0.563841      0.000000      0.239801      256_128_64_32   

  embedding_dims  \
0          16_16   
1          16_16   
2          16_16   

                

In [ ]:
tracking_df.drop(columns=['best_model_path','minutes_per_epoch'])

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,False,20380,32,8,13,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
1,2,False,6793,32,19,10,0.001,0.569713,0.528593,0.000000,0.247884,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.
2,2,False,20380,32,9,8,0.001,0.584609,0.563841,0.000000,0.239801,256_128_64_32,16_16,Increased frac from 0.1 to 0.3.
